## Notebook 21 - Unknown Inference

Apply the trained nb17 model (2d_resnet_v4.keras) to all 2391 unknown recordings.
Output: outputs/unknown_predictions.csv with filename, predicted class, and probability.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import spectrogram as stft_spec
from scipy.fft import dct as scipy_dct
import tensorflow as tf
from tensorflow import keras

PROJECT_ROOT = Path(r"D:/sop")
DECOMP_DIR   = PROJECT_ROOT / "data" / "decomposed" / "unknown"
PROC_DIR     = PROJECT_ROOT / "data" / "processed"  / "unknown"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
MODELS_DIR   = PROJECT_ROOT / "models"

FS          = 1000
SEGMENT_LEN = 4000
STRIDE      = 500
NPERSEG     = 128
NOVERLAP    = 96
FREQ_BINS   = 64
TIME_FRAMES = 122
N_CHANNELS  = 5

print(f"TF {tf.__version__}")
print(f"Unknown decomposed files : {len(list(DECOMP_DIR.glob('*.npz')))}")
print(f"Unknown processed files  : {len(list(PROC_DIR.glob('*.npy')))}")


TF 2.19.0
Unknown decomposed files : 2391
Unknown processed files  : 2391


In [2]:
# build mel filterbank once
def _hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + hz / 700.0)

def _mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

def _build_mel_filterbank(n_mels, n_fft, sr, fmin=20.0):
    fmax    = sr / 2.0
    mel_pts = np.linspace(_hz_to_mel(fmin), _hz_to_mel(fmax), n_mels + 2)
    hz_pts  = _mel_to_hz(mel_pts)
    bins    = np.floor((n_fft + 1) * hz_pts / sr).astype(int)
    fb = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float32)
    for m in range(1, n_mels + 1):
        lo, mid, hi = bins[m-1], bins[m], bins[m+1]
        for k in range(lo, mid):
            fb[m-1, k] = (k - lo) / max(mid - lo, 1)
        for k in range(mid, hi):
            fb[m-1, k] = (hi - k) / max(hi - mid, 1)
    return fb

_MEL_FB = _build_mel_filterbank(n_mels=FREQ_BINS, n_fft=NPERSEG, sr=FS)
print(f"Mel filterbank: {_MEL_FB.shape}")


Mel filterbank: (64, 65)


In [3]:
def _norm(arr):
    mu, sigma = arr.mean(), arr.std() + 1e-8
    return ((arr - mu) / sigma).astype(np.float32)


def compute_mel_spec(sig_1d):
    _, _, S = stft_spec(sig_1d.astype(np.float32),
                        fs=FS, nperseg=NPERSEG, noverlap=NOVERLAP)
    mel_S   = _MEL_FB @ np.abs(S)
    log_mel = 10.0 * np.log10(mel_S + 1e-10)
    return _norm(log_mel[:, :TIME_FRAMES])


def compute_mfcc(sig_1d):
    _, _, S = stft_spec(sig_1d.astype(np.float32),
                        fs=FS, nperseg=NPERSEG, noverlap=NOVERLAP)
    mel_S   = _MEL_FB @ np.abs(S)
    log_mel = np.log(mel_S + 1e-10)
    mfcc    = scipy_dct(log_mel, axis=0, norm='ortho')
    return _norm(mfcc[:, :TIME_FRAMES])


def compute_rms_map(sig_1d):
    hop      = NPERSEG - NOVERLAP
    n_frames = (len(sig_1d) - NPERSEG) // hop + 1
    rms = np.array([
        np.sqrt(np.mean(sig_1d[i*hop : i*hop+NPERSEG].astype(np.float32)**2))
        for i in range(n_frames)
    ], dtype=np.float32)
    rms = rms[:TIME_FRAMES]
    if len(rms) < TIME_FRAMES:
        rms = np.pad(rms, (0, TIME_FRAMES - len(rms)), mode='edge')
    rms_map = np.tile(rms[np.newaxis, :], (FREQ_BINS, 1))
    return _norm(rms_map)


def compute_log_spec(sig_1d):
    _, _, S = stft_spec(sig_1d.astype(np.float32),
                        fs=FS, nperseg=NPERSEG, noverlap=NOVERLAP)
    log_S = 10.0 * np.log10(S + 1e-10)
    return _norm(log_S[1:FREQ_BINS+1, :TIME_FRAMES])


def window_to_image(seg_raw, seg_u):
    ch0 = compute_mel_spec(seg_raw)
    ch1 = compute_mfcc(seg_raw)
    ch2 = compute_rms_map(seg_raw)
    ch3 = compute_log_spec(seg_u[0] + seg_u[1])
    ch4 = compute_log_spec(seg_u[2] + seg_u[3])
    return np.stack([ch0, ch1, ch2, ch3, ch4], axis=-1)


In [ ]:
model = keras.models.load_model(MODELS_DIR / "2d_resnet_v4.keras")
print(f"Model loaded: {model.name}")
print(f"Parameters  : {model.count_params():,}")


Model loaded: SpectrogramResNet_v4
Parameters  : 2,444,577


In [5]:
def load_unknown_recording(stem):
    # stem is like 'u10_AV_decomposed'
    raw_stem = stem.replace("_decomposed", "")
    raw = np.load(PROC_DIR  / f"{raw_stem}.npy")
    u   = np.load(DECOMP_DIR / f"{stem}.npz")["u"]
    N   = min(len(raw), u.shape[1])
    if N < SEGMENT_LEN:
        raw = np.pad(raw, (0, SEGMENT_LEN - N), mode="reflect")
        u   = np.pad(u, ((0,0),(0, SEGMENT_LEN - N)), mode="reflect")
        N   = SEGMENT_LEN
    return raw[:N], u[:, :N], N


def predict_recording(stem):
    raw, u, N = load_unknown_recording(stem)
    images, start = [], 0
    while True:
        end = start + SEGMENT_LEN
        if end > N:
            r_pad = np.pad(raw[start:N],  (0, end-N), mode="reflect")
            u_pad = np.pad(u[:, start:N], ((0,0),(0,end-N)), mode="reflect")
            images.append(window_to_image(r_pad, u_pad))
            break
        images.append(window_to_image(raw[start:end], u[:, start:end]))
        start += STRIDE
        if start >= N:
            break
    X     = np.array(images, dtype=np.float32)
    probs = model.predict(X, batch_size=32, verbose=0).ravel()
    return float(probs.mean())


In [6]:
# collect all decomposed stems
all_stems = sorted([p.stem for p in DECOMP_DIR.glob("*.npz")])
print(f"Running inference on {len(all_stems)} unknown recordings...")

results = []
failed  = []

for i, stem in enumerate(all_stems):
    try:
        prob = predict_recording(stem)
        pred = "present" if prob >= 0.46 else "absent"
        results.append({"file": stem, "predicted_class": pred, "probability": round(prob, 4)})
    except Exception as e:
        failed.append({"file": stem, "error": str(e)})

    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{len(all_stems)} done")

print(f"\nComplete. Successes: {len(results)}  Failures: {len(failed)}")

Running inference on 2391 unknown recordings...
  200/2391 done
  400/2391 done
  600/2391 done
  800/2391 done
  1000/2391 done
  1200/2391 done
  1400/2391 done
  1600/2391 done
  1800/2391 done
  2000/2391 done
  2200/2391 done

Complete. Successes: 2391  Failures: 0


In [7]:
df_out = pd.DataFrame(results)

n_present = (df_out["predicted_class"] == "present").sum()
n_absent  = (df_out["predicted_class"] == "absent").sum()

print(f"Predicted present : {n_present} ({n_present/len(df_out)*100:.1f}%)")
print(f"Predicted absent  : {n_absent}  ({n_absent/len(df_out)*100:.1f}%)")
print(f"Mean probability  : {df_out['probability'].mean():.3f}")
print()
print(df_out.head(10).to_string(index=False))

out_path = OUTPUTS_DIR / "unknown_predictions.csv"
df_out.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")


Predicted present : 1573 (65.8%)
Predicted absent  : 818  (34.2%)
Mean probability  : 0.577

                       file predicted_class  probability
          u10_AV_decomposed          absent       0.3529
u10_AV_gauss1093_decomposed         present       0.5336
u10_AV_gauss1249_decomposed         present       0.5801
u10_AV_gauss1405_decomposed         present       0.5367
u10_AV_gauss1561_decomposed         present       0.5454
 u10_AV_gauss157_decomposed         present       0.5606
u10_AV_gauss1717_decomposed         present       0.6432
u10_AV_gauss1873_decomposed         present       0.5843
   u10_AV_gauss1_decomposed         present       0.5842
u10_AV_gauss2029_decomposed         present       0.5492

Saved: D:\sop\outputs\unknown_predictions.csv


In [8]:
# high confidence predictions (prob >= 0.80 or prob <= 0.20)
high_conf = df_out[
    (df_out["probability"] >= 0.80) | (df_out["probability"] <= 0.20)
].copy()

print(f"High confidence (prob>=0.80 or <=0.20): {len(high_conf)} / {len(df_out)}")
print(f"  Confident present : {(high_conf['predicted_class']=='present').sum()}")
print(f"  Confident absent  : {(high_conf['predicted_class']=='absent').sum()}")
print()

# borderline (uncertain zone 0.40-0.60)
borderline = df_out[
    (df_out["probability"] >= 0.40) & (df_out["probability"] <= 0.60)
]
print(f"Borderline (0.40-0.60): {len(borderline)} / {len(df_out)}")


High confidence (prob>=0.80 or <=0.20): 947 / 2391
  Confident present : 676
  Confident absent  : 271

Borderline (0.40-0.60): 570 / 2391


In [9]:
import re

# patient-level aggregation
df_out['patient'] = df_out['file'].apply(
    lambda s: re.match(r'u(\d+)_', s).group(1)
)

pat = (df_out.groupby('patient')
             .agg(mean_prob=('probability', 'mean'),
                  n_recordings=('file', 'count'))
             .reset_index())

pat['predicted_class'] = pat['mean_prob'].apply(
    lambda p: 'present' if p >= 0.61 else 'absent'
)

n_pat_present = (pat['predicted_class'] == 'present').sum()
n_pat_absent  = (pat['predicted_class'] == 'absent').sum()

print(f"Patients found      : {len(pat)}")
print(f"Avg recordings/pat  : {pat['n_recordings'].mean():.1f}")
print(f"Predicted present   : {n_pat_present} ({n_pat_present/len(pat)*100:.1f}%)")
print(f"Predicted absent    : {n_pat_absent}  ({n_pat_absent/len(pat)*100:.1f}%)")
print()
print(pat.head(10).to_string(index=False))

pat_path = OUTPUTS_DIR / "unknown_patient_predictions.csv"
pat.to_csv(pat_path, index=False)
print(f"\nSaved: {pat_path}")

Patients found      : 68
Avg recordings/pat  : 35.2
Predicted present   : 30 (44.1%)
Predicted absent    : 38  (55.9%)

patient  mean_prob  n_recordings predicted_class
      1   0.461984            64          absent
     10   0.608084            32          absent
     11   0.871275            32         present
     12   0.510275            32          absent
     13   0.902988            16         present
     14   0.553528            32          absent
     15   0.476352            64          absent
     16   0.457247            32          absent
     17   0.454631            16          absent
     18   0.554567            64          absent

Saved: D:\sop\outputs\unknown_patient_predictions.csv


In [ ]:
# clean patient summary CSV
summary = pat.copy()
summary['patient_id'] = summary['patient'].astype(int)
summary['murmur_probability'] = summary['mean_prob'].round(4)

def confidence_label(p):
    if p >= 0.80 or p <= 0.20:
        return 'high'
    elif p >= 0.40 and p <= 0.60:
        return 'borderline'
    else:
        return 'moderate'

summary['confidence'] = summary['murmur_probability'].apply(confidence_label)

summary = (summary[['patient_id', 'n_recordings', 'murmur_probability', 'confidence', 'predicted_class']]
           .sort_values('patient_id')
           .reset_index(drop=True))

print(summary.to_string(index=False))

summary_path = OUTPUTS_DIR / "unknown_patient_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"\nSaved: {summary_path}")


 patient_id  n_recordings  murmur_probability confidence predicted_class
          1            64              0.4620 borderline          absent
          2            15              0.9786       high         present
          3            30              0.7262   moderate         present
          4            30              0.7174   moderate         present
          5            15              0.2164   moderate          absent
          6            30              0.8252       high         present
          7            15              0.8424       high         present
          8            30              0.8074       high         present
          9            30              0.7777   moderate         present
         10            32              0.6081   moderate          absent
         11            32              0.8713       high         present
         12            32              0.5103 borderline          absent
         13            16              0.9030      